In [ ]:
import pandas as pd

industry = pd.read_csv("/kaggle/input/datasets/bharathwaajg/energys/industry.csv")
solar = pd.read_csv("/kaggle/input/datasets/bharathwaajg/energys/solar.csv")


In [ ]:
print("Industry Columns:")
print(industry.columns)

print("\nSolar Columns:")
print(solar.columns)


In [ ]:
industry["Date"] = pd.to_datetime(industry["Date"], errors="coerce")
solar["date"] = pd.to_datetime(solar["date"], dayfirst=True, errors="coerce")


In [ ]:
industry.head()

In [ ]:
industry.head()
solar.head()


In [ ]:
solar = solar.groupby("date")["E-Today(KWH)"].mean().reset_index()


In [ ]:
print("Solar rows:", len(solar))


In [ ]:
solar.rename(columns={"date": "Date"}, inplace=True)


In [ ]:
print(solar.columns)
print(industry.columns)


In [ ]:
industry["Date"] = pd.to_datetime(industry["Date"], errors="coerce")
solar["Date"] = pd.to_datetime(solar["Date"], dayfirst=True, errors="coerce")


In [ ]:
solar.rename(columns={"E-Today(KWH)": "E-Today (KWH)"}, inplace=True)

In [ ]:
print(solar.columns)
print(industry.columns)

In [ ]:
final_df = pd.merge(
    industry,
    solar[["Date", "E-Today (KWH)"]],
    on="Date",
    how="inner"
)


In [ ]:
final_df.to_csv('final data.csv',index=False)

In [ ]:
print(final_df.shape)
print(final_df.head())


In [ ]:
print(final_df.columns)

In [ ]:
final_df = final_df.drop(columns=["Unnamed: 17"], errors="ignore")


In [ ]:
needed_columns = [
    "Date",
    "Daily Energy consumption in units",
    "E-Today (KWH)",
    "Diesel Consumed in ltrs",
    "Diesel cost / Day in INR",
    "Clutch Produced",
    "Power cut in hrs",
    "DG Units"
]

final_df = final_df[needed_columns]


In [ ]:
print(final_df.head())
print(final_df.shape)


In [ ]:
final_df.isnull().sum()


In [ ]:
columns_to_fill = [
    "Daily Energy consumption in units",
    "Diesel cost / Day in INR", 
    "Power cut in hrs",
    "DG Units"
]
for col in columns_to_fill:
    final_df[col] = final_df[col].fillna(0)  # Or .ffill()


In [ ]:
final_df["Clutch Produced"] = final_df["Clutch Produced"].fillna(final_df["Clutch Produced"].median())


In [ ]:
final_df["Diesel Consumed in ltrs"] = final_df["Diesel Consumed in ltrs"].fillna(0)


In [ ]:
final_df.isnull().sum()


In [ ]:
final_df = final_df.sort_values("Date")


In [ ]:
solar_series = final_df.sort_values("Date")["E-Today (KWH)"]


In [ ]:
solar_series.index = pd.to_datetime(final_df.sort_values("Date")["Date"])


In [ ]:
print("\nSolar production stats:") 
print(solar_series.describe())

In [ ]:
final_df["Solar_scaled"] = final_df["E-Today (KWH)"] * 1000


In [ ]:
final_df[[
    "Daily Energy consumption in units",
    "Solar_scaled"
]].describe()


In [ ]:
solar_series = final_df.sort_values("Date")["Solar_scaled"]
solar_series.index = pd.to_datetime(final_df.sort_values("Date")["Date"])


In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(solar_series)

print("ADF Statistic:", result[0])
print("p-value:", result[1])


In [ ]:
solar_diff = solar_series.diff().dropna()


In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(solar_diff)

print("ADF Statistic:", result[0])
print("p-value:", result[1])


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(
    solar_series,
    order=(1,1,1),
    seasonal_order=(1,1,1,30)
)

model_fit = model.fit()

forecast = model_fit.forecast(steps=30)

print(forecast)

In [ ]:
forecast_df = forecast.reset_index()
forecast_df.columns = ["Date", "Forecast_Renewable_KWH"]

print(forecast_df.head())


In [ ]:
avg_demand = final_df["Daily Energy consumption in units"].mean()
print("Average Industrial Demand:", avg_demand)


In [ ]:
forecast_df["Renewable_Coverage_%"] = (
    forecast_df["Forecast_Renewable_KWH"] / 43986.3477434679
) * 100

forecast_df.head()


In [ ]:
def decision(coverage):
    if coverage >= 70:
        return "Increase Production - High Renewable Availability"
    elif coverage >= 50:
        return "Normal Operation - Moderate Renewable"
    else:
        return "Limit Heavy Machinery - Use Grid/Diesel Carefully"

forecast_df["Suggested_Action"] = forecast_df["Renewable_Coverage_%"].apply(decision)

forecast_df.head()


In [ ]:
model_fit.save("sarima_model.pkl")

In [ ]:
final_df.to_csv("final_energy_dataset.csv", index=False)


In [ ]:
avg_demand = 43986.3477434679

plt.figure(figsize=(14,6))

plt.plot(forecast.index, forecast, label="Forecast Renewable")
plt.axhline(y=avg_demand, linestyle='--', label="Average Industrial Demand")

plt.title("Renewable Forecast vs Industrial Demand")
plt.xlabel("Date")
plt.ylabel("Energy (KWH)")
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
industry['Daily Energy consumption in units'].mean()
